In [5]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

print(project_root)


c:\Users\Admin\OneDrive\Desktop\projects\spotify-candidate-quality


In [ ]:
# WHY:
# We load the processed dataset (after EDA + target creation) to ensure
# feature engineering always starts from a frozen, decision-approved dataset.
# This prevents accidental changes to target logic during experimentation.

from src.config import PROCESSED_DATA_DIR
import pandas as pd

DATA_PATH = PROCESSED_DATA_DIR / "spotify_labeled.parquet"
df = pd.read_parquet(DATA_PATH)

df.head()


,genre,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,...,liveness,loudness,mode,speechiness,tempo,time_signature,valence,pop_pct_genre,low_performance,high_skip_risk_proxy
0,Movie,Henri Salvador,C'est beau de faire un Show,0BRjO6ga9RKCKjfDqeFgWV,0,0.611,0.389,99373,0.910,0.000,...,0.3460,-1.828,Major,0.0525,166.969,4/4,0.814,0.111068,1,1
1,Movie,Martin & les fées,Perdu d'avance (par Gad Elmaleh),0BjC1NfoEOOusryehmNudP,1,0.246,0.590,137373,0.737,0.000,...,0.1510,-5.559,Minor,0.0868,174.003,4/4,0.816,0.260377,0,0
2,Movie,Joseph Williams,Don't Let Me Be Lonely Tonight,0CoSDzoNIKCRs124s9uTVy,3,0.952,0.663,170267,0.131,0.000,...,0.1030,-13.879,Minor,0.0362,99.488,5/4,0.368,0.369523,0,0
3,Movie,Henri Salvador,Dis-moi Monsieur Gordon Cooper,0Gc6TVm52BwZD07Ki6tIvf,0,0.703,0.240,152427,0.326,0.000,...,0.0985,-12.178,Major,0.0395,171.758,4/4,0.227,0.111068,1,1
4,Movie,Fabien Nataf,Ouverture,0IuslXpMROHdEPvSl1fTQK,4,0.950,0.331,82625,0.225,0.123,...,0.2020,-21.150,Major,0.0456,140.576,4/4,0.390,0.405137,0,0


In [ ]:
# WHY:
# Raw audio features represent intrinsic musical properties of a track.
# We keep them untransformed so the model can learn absolute signals
# (e.g., loud tracks vs quiet tracks), not only relative deviations.

audio_features = [
    'acousticness',
    'danceability',
    'energy',
    'loudness',
    'tempo',
    'speechiness',
    'instrumentalness',
    'liveness',
    'valence',
    'duration_ms'
]


In [ ]:
# WHY:
# These features describe musical structure rather than sound texture.
# They are kept numeric to avoid unnecessary one-hot expansion
# and are suitable for both linear and tree-based models.

categorical_numeric = [
    "mode"
]


In [ ]:
# WHY:
# Time signature is categorical but ordinal-like.
# Converting it into a numeric-friendly representation allows the model
# to capture rhythmic structure without exploding dimensionality.
df["time_signature_num"] = (
    df["time_signature"]
    .str.split("/")
    .str[0]
    .astype(int)
)


In [9]:
categorical_numeric.append("time_signature_num")


In [10]:
df = df.drop(columns=["time_signature"])


In [ ]:
# WHY:
# Certain audio features (e.g., loudness, energy) have very different
# baselines across genres. Normalizing them within genre allows the model
# to learn relative deviation from genre norms rather than absolute scale.
normalize_features = [
    'loudness',
    'energy',
    'danceability',
    'tempo',
    'speechiness'
]


In [ ]:
# WHY:
# Genre-level mean and standard deviation are computed to create
# context-aware z-score features, aligning feature space with
# our genre-relative target definition.
genre_stats = (
    df
    .groupby("genre")[normalize_features]
    .agg(["mean", "std"])
)

genre_stats.head()



loudness              energy           danceability  \
                       mean       std      mean       std         mean   
genre                                                                    
Alternative       -6.540803  2.764235  0.711519  0.206063     0.541898   
Anime             -7.917802  6.184689  0.665356  0.299668     0.472090   
Blues             -9.053807  3.855575  0.606171  0.229498     0.528232   
Children's Music  -8.399591  4.201918  0.593204  0.253964     0.598829   
Classical        -21.544477  7.682214  0.177984  0.225483     0.305958   

                                 tempo            speechiness            
                       std        mean        std        mean       std  
genre                                                                    
Alternative       0.150391  122.534485  30.129437    0.088783  0.091241  
Anime             0.149229  126.629156  33.160207    0.065102  0.053889  
Blues             0.145147  121.137637  30.387693    0.061809  0.061537  
Children's Music  0.168058  121.638247  30.144981    0.097763  0.123095  
Classical         0.135201  104.341807  31.038727    0.052001  0.040798

In [ ]:
# WHY:
# Z-score normalization within genre captures how unusual a track is
# relative to its genre peers (e.g., unusually loud Jazz track),
# which is more informative than raw values for risk estimation.
for col in normalize_features:
    mean_col = df["genre"].map(genre_stats[col]["mean"])
    std_col  = df["genre"].map(genre_stats[col]["std"])

    df[f"{col}_z_genre"] = (df[col] - mean_col) / std_col


In [ ]:
# WHY:
# We inspect summary statistics to verify that genre-normalized features
# are well-behaved (approximately centered, no extreme instability),
# ensuring they are safe to use for modeling.
df[[f"{c}_z_genre" for c in normalize_features]].describe()


,loudness_z_genre,energy_z_genre,danceability_z_genre,tempo_z_genre,speechiness_z_genre
count,2.326060e+05,2.326060e+05,2.326060e+05,2.326060e+05,2.326060e+05
mean,4.527073e-17,4.139125e-18,-1.600665e-17,-3.476254e-17,-3.338793e-17
std,9.999484e-01,9.999484e-01,9.999484e-01,9.999484e-01,9.999484e-01
min,-1.026266e+01,-4.651054e+00,-4.403636e+00,-3.383603e+00,-4.033315e+00
25%,-5.390117e-01,-7.118165e-01,-6.807774e-01,-7.946487e-01,-5.014722e-01
50%,1.714438e-01,3.160411e-02,3.529424e-02,-8.119783e-02,-3.010812e-01
75%,7.026695e-01,7.772408e-01,7.149183e-01,6.741563e-01,1.802518e-01
max,3.319299e+00,4.526383e+00,3.905599e+00,4.466004e+00,3.217774e+01


In [ ]:
# WHY:
# Artist-level audio aggregates provide prior context about an artist's
# typical sound without using popularity or target information.
# This helps the model generalize better, especially for sparse tracks,
# while remaining leakage-safe.

artist_audio_means = (
    df.groupby('artist_name')[audio_features]
      .mean()
      .add_prefix('artist_avg_')
)

df = df.join(artist_audio_means, on='artist_name')


In [ ]:
# WHY:
# Genre is treated as recommendation context, not intrinsic truth.
# One-hot encoding preserves interpretability and avoids target leakage,
# which could occur with target encoding since the label is genre-relative.
df = pd.get_dummies(df, columns=['genre'], prefix='genre')


In [ ]:
# WHY:
# We assert that the original genre column is removed to prevent
# accidental reuse or inconsistent feature handling downstream.
assert "genre" not in df.columns

In [ ]:
# WHY:
# This check ensures genre encoding did not introduce extreme imbalance
# or missing categories, which could bias the model.
genre_cols = [c for c in df.columns if c.startswith("genre_")]
len(genre_cols), genre_cols[:10]


(25,
 ['genre_Alternative',
  'genre_Anime',
  'genre_Blues',
  "genre_Children's Music",
  'genre_Classical',
  'genre_Comedy',
  'genre_Country',
  'genre_Dance',
  'genre_Electronic',
  'genre_Folk'])

In [19]:
df[genre_cols].mean().sort_values().head()


genre_Movie      0.033559
genre_Opera      0.035597
genre_Country    0.037248
genre_Dance      0.037407
genre_Reggae     0.037708
dtype: float64

In [22]:
df["key"].dtype


dtype('O')

In [ ]:
# WHY:
# We map musical key to a numeric scale to enable cyclic encoding,
# preserving harmonic adjacency between keys.
key_mapping = {
    "C": 0,
    "C#": 1, "Db": 1,
    "D": 2,
    "D#": 3, "Eb": 3,
    "E": 4,
    "F": 5,
    "F#": 6, "Gb": 6,
    "G": 7,
    "G#": 8, "Ab": 8,
    "A": 9,
    "A#": 10, "Bb": 10,
    "B": 11
}

df["key_num"] = df["key"].map(key_mapping)


In [24]:
df["key_num"].isna().sum()


0

In [ ]:
# WHY:
# Cyclic encoding preserves circular relationships in musical key,
# preventing arbitrary linear ordering (e.g., key 11 far from key 0).
# This is especially beneficial for linear models.
df["key_sin"] = np.sin(2 * np.pi * df["key_num"] / 12)
df["key_cos"] = np.cos(2 * np.pi * df["key_num"] / 12)


In [26]:
df[["key_sin", "key_cos"]].describe()


,key_sin,key_cos
count,232606.000000,2.326060e+05
mean,-0.025995,4.666267e-02
std,0.675130,7.357651e-01
min,-1.000000,-1.000000e+00
25%,-0.500000,-8.660254e-01
50%,0.000000,-1.836970e-16
75%,0.500000,8.660254e-01
max,1.000000,1.000000e+00


In [27]:
categorical_numeric.extend(["key_sin", "key_cos"])


In [29]:
df = df.drop(columns=["key", "key_num"])


In [30]:
assert "key" not in df.columns
assert "key_num" not in df.columns


In [ ]:
# WHY:
# We explicitly define the final feature list to:
# - Prevent accidental leakage
# - Ensure reproducibility
# - Make the modeling contract explicit and reviewable
feature_cols = (
    audio_features +
    [f"{c}_z_genre" for c in normalize_features] +
    list(artist_audio_means.columns) +
    categorical_numeric +
    genre_cols
)


In [32]:
# 1. No target leakage
leakage_cols = [
    "popularity",
    "pop_pct_genre",
    "low_performance",
    "high_skip_risk_proxy"
]

for col in leakage_cols:
    assert col not in feature_cols, f"Leakage detected: {col}"

# 2. All features exist in df
missing = set(feature_cols) - set(df.columns)
assert len(missing) == 0, f"Missing features: {missing}"

# 3. No duplicates
assert len(feature_cols) == len(set(feature_cols)), "Duplicate features found"

len(feature_cols)


54